# Prepare matched Tractor-Mix / SAIGE pilot inputs

> This exploratory copy is retained for provenance. Use
> `notebooks/tractor_01_prepare_inputs.ipynb` as the canonical current workflow.

The current preparation workflow builds a shared **recommended-model-complete**
cohort from `covariates.source_rebuilt.csv.gz` and writes `pheno_cov.tsv`,
standalone `covariates_limited.tsv` / `covariates_full.tsv`, and matching
`covariate_columns_{limited,full}.txt` files. “Full” means limited +
standardized coverage + GC dummies; it excludes extraction, platform,
methylation caller, and SV counts.

FLARE VCF URIs are resolved from the Terra data table `aou_lr_chrom` via the
firecloud API (column `model_chr_anc_vcf`). Workspace:
`allofus-drc-wgs-LR-prodData` / `AoU_DRC_LongReads_PhaseTwo_Storage`.

Phenotypes: `gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scratch/kvg/saige/AoU_Phase2_Phenotype.csv.gz`

Confirmed FLARE ancestries: `eas=0,amr=1,eur=2,afr=3,sas=4` → `num_ancs=5`.

Set `TRACTOR_COVARIATES_GCS` to the uploaded
`covariates.source_rebuilt.csv.gz` before running the canonical notebook.

In [15]:
!gsutil ls $WORKSPACE_BUCKET/{scripts,tractor_mix_pilot,notebooks}

gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/
gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/build_saige_plink_and_grm.sh
gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/compare_calibration.py
gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/fit_null_and_score.R
gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/fit_saige_null.R
gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/make_plink_keep.py
gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/plot_tractor_results.py
gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/reorder_dosages.py
gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/resolve_flare_uris.py
gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/run_saige_step2.R
gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/select_phenotypes.py
gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/sparsify_grm.R
gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09

In [13]:
!find . -name 'covariates*.txt'

In [1]:
# Sync pipeline scripts from the workspace bucket onto this VM
import os
from pathlib import Path

ws = os.environ.get("WORKSPACE_BUCKET", "")
assert ws, "WORKSPACE_BUCKET is unset; run this notebook in an AoU/Terra runtime"

scripts_dir = Path("../scripts")
scripts_dir.mkdir(parents=True, exist_ok=True)
# Canonical staging path (same prefix as WDL script inputs).
# Fall back to legacy $WORKSPACE_BUCKET/scripts/ if needed.
src = f"{ws}/tractor_mix/scripts"
rc = get_ipython().system(f"gsutil -q ls {src}/select_phenotypes.py")
if rc != 0:
    src = f"{ws}/scripts"
    print(f"NOTE: using legacy script prefix {src}")
get_ipython().system(f"gsutil -m cp {src}/* {scripts_dir}/")
select_py = scripts_dir / "select_phenotypes.py"
text = select_py.read_text()
assert "covariates_limited.tsv" in text, (
    f"{select_py} is outdated (no covariates_limited.tsv). "
    f"Upload the current tractor_mix/scripts/* to {ws}/tractor_mix/scripts/ "
    "then re-run this cell."
)
print("scripts synced from", src)
print("scripts:", sorted(p.name for p in scripts_dir.glob("*")))

CommandException: One or more URLs matched no objects.
NOTE: using legacy script prefix gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts
Copying gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/build_saige_plink_and_grm.sh...
Copying gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/compare_calibration.py...
Copying gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/fit_null_and_score.R...
Copying gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/make_plink_keep.py...
Copying gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/fit_saige_null.R...
Copying gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/plot_tractor_results.py...
Copying gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/reorder_dosages.py...
Copying gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/run_saige_step2.R...
Copying gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scripts/select_phenotypes.py...
Copying gs://fc-secure

In [2]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

ROOT = Path("..").resolve()
SCRIPTS = ROOT / "scripts"
sys.path.insert(0, str(SCRIPTS))

from resolve_flare_uris import (
    DEFAULT_ENTITY_TYPE,
    DEFAULT_NAMESPACE,
    DEFAULT_WORKSPACE,
    fetch_chrom_table_firecloud,
    resolve_uris,
)

OUT_DIR = Path("tractor_mix_pilot_inputs")
OUT_DIR.mkdir(exist_ok=True)

# Terra workspace hosting the aou_lr_chrom data table
TERRA_NAMESPACE = os.environ.get("TERRA_NAMESPACE", DEFAULT_NAMESPACE)
TERRA_WORKSPACE = os.environ.get("TERRA_WORKSPACE", DEFAULT_WORKSPACE)
TERRA_ENTITY_TYPE = os.environ.get("TERRA_ENTITY_TYPE", DEFAULT_ENTITY_TYPE)

print(f"Fetching {TERRA_ENTITY_TYPE} from {TERRA_NAMESPACE}/{TERRA_WORKSPACE} ...")
chrom_table = fetch_chrom_table_firecloud(
    TERRA_NAMESPACE, TERRA_WORKSPACE, TERRA_ENTITY_TYPE
)
print(f"Loaded {len(chrom_table)} chromosomes: {sorted(chrom_table)}")

uris = resolve_uris(
    chrom_table,
    scan_chrom="chr22",
    grm_chroms=["chr1", "chr22"],
    uri_column="model_chr_anc_vcf",
)
(OUT_DIR / "flare_uris.json").write_text(json.dumps(uris, indent=2) + "\n")

FLARE_VCF = os.environ.get("TRACTOR_FLARE_VCF", uris["flare_vcf"])
GRM_VCFS = uris["grm_vcfs"]

PHENOTYPE_GCS = os.environ.get(
    "TRACTOR_PHENOTYPE_GCS",
    "gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scratch/kvg/saige/AoU_Phase2_Phenotype.csv.gz",
)
PHENOTYPE_CSV = Path("AoU_Phase2_Phenotype.csv.gz")
COVARIATES_GCS = os.environ.get("TRACTOR_COVARIATES_GCS", "")
COVARIATES_CSV = Path("covariates.source_rebuilt.csv.gz")
# ##ANCESTRY=<eas=0,amr=1,eur=2,afr=3,sas=4>
NUM_ANCS = int(os.environ.get("TRACTOR_NUM_ANCS", "5"))
N_PHENOTYPES = 10
MIN_CASES = 100
N_PCS = 10

print("FLARE_VCF (chr22 scan):", FLARE_VCF)
print("GRM_VCFS (chr1+chr22):")
for u in GRM_VCFS:
    print(" ", u)
print("PHENOTYPE_GCS:", PHENOTYPE_GCS)
print("COVARIATES_CSV:", COVARIATES_CSV)
print("NUM_ANCS:", NUM_ANCS)
print("OUT_DIR:", OUT_DIR.resolve())
display(uris)

Fetching aou_lr_chrom from allofus-drc-wgs-LR-prodData/AoU_DRC_LongReads_PhaseTwo_Storage ...
Loaded 22 chromosomes: ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21', 'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9']
FLARE_VCF (chr22 scan): gs://fc-secure-1c6b9393-5e5d-4a87-b483-d1f0b019af92/submissions/aac26765-3585-4399-b888-ff8521112d20/flare/b13bb7e0-4684-4b4a-a8b3-a4d94e7f48c4/call-flare_task/aou_lr_phase2_v1.chr22.anc.vcf.gz
GRM_VCFS (chr1+chr22):
  gs://fc-secure-1c6b9393-5e5d-4a87-b483-d1f0b019af92/submissions/c70fc078-7d46-4da1-8f59-30a7ebd4b738/flare/81ea1db7-94da-4a6b-b869-35b292d2f9a1/call-flare_task/aou_lr_phase2_v1.chr1.anc.vcf.gz
  gs://fc-secure-1c6b9393-5e5d-4a87-b483-d1f0b019af92/submissions/aac26765-3585-4399-b888-ff8521112d20/flare/b13bb7e0-4684-4b4a-a8b3-a4d94e7f48c4/call-flare_task/aou_lr_phase2_v1.chr22.anc.vcf.gz
PHENOTYPE_GCS: gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececea

{'flare_vcf': 'gs://fc-secure-1c6b9393-5e5d-4a87-b483-d1f0b019af92/submissions/aac26765-3585-4399-b888-ff8521112d20/flare/b13bb7e0-4684-4b4a-a8b3-a4d94e7f48c4/call-flare_task/aou_lr_phase2_v1.chr22.anc.vcf.gz',
 'grm_vcfs': ['gs://fc-secure-1c6b9393-5e5d-4a87-b483-d1f0b019af92/submissions/c70fc078-7d46-4da1-8f59-30a7ebd4b738/flare/81ea1db7-94da-4a6b-b869-35b292d2f9a1/call-flare_task/aou_lr_phase2_v1.chr1.anc.vcf.gz',
  'gs://fc-secure-1c6b9393-5e5d-4a87-b483-d1f0b019af92/submissions/aac26765-3585-4399-b888-ff8521112d20/flare/b13bb7e0-4684-4b4a-a8b3-a4d94e7f48c4/call-flare_task/aou_lr_phase2_v1.chr22.anc.vcf.gz'],
 'scan_chrom': 'chr22',
 'grm_chroms': ['chr1', 'chr22'],
 'uri_column': 'model_chr_anc_vcf'}

In [3]:
def sh(cmd: str) -> None:
    print(cmd)
    rc = get_ipython().system(cmd)
    if rc:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")


ws = os.environ.get("WORKSPACE_BUCKET", "")

# Pull phenotype / source-rebuilt covariates into the notebook cwd if needed.
if not COVARIATES_CSV.exists() and COVARIATES_GCS:
    sh(f"gsutil cp {COVARIATES_GCS} {COVARIATES_CSV}")

if not PHENOTYPE_CSV.exists():
    sh(f"gsutil cp {PHENOTYPE_GCS} {PHENOTYPE_CSV}")

assert COVARIATES_CSV.exists(), (
    f"Missing {COVARIATES_CSV}. Upload the rebuilt table and set "
    "TRACTOR_COVARIATES_GCS=gs://.../covariates.source_rebuilt.csv.gz, "
    "or place it in the notebook working directory."
)
assert PHENOTYPE_CSV.exists(), f"Missing {PHENOTYPE_CSV}"

# Sample IDs from FLARE VCF (stream; do not copy full VCF locally)
vcf_samples = OUT_DIR / "vcf_samples.txt"
if FLARE_VCF.startswith("gs://"):
    sh(f"gsutil cat {FLARE_VCF} | bcftools query -l > {vcf_samples}")
else:
    sh(f"bcftools query -l {FLARE_VCF} > {vcf_samples}")
print(f"Wrote {vcf_samples} ({sum(1 for _ in open(vcf_samples)):,} samples)")

gsutil cat gs://fc-secure-1c6b9393-5e5d-4a87-b483-d1f0b019af92/submissions/aac26765-3585-4399-b888-ff8521112d20/flare/b13bb7e0-4684-4b4a-a8b3-a4d94e7f48c4/call-flare_task/aou_lr_phase2_v1.chr22.anc.vcf.gz | bcftools query -l > tractor_mix_pilot_inputs/vcf_samples.txt
Exception ignored in: <_io.TextIOWrapper name='<stdout>' mode='w' encoding='utf-8'>
BrokenPipeError: [Errno 32] Broken pipe
Wrote tractor_mix_pilot_inputs/vcf_samples.txt (12,347 samples)


In [7]:
!ls tractor_mix_pilot_inputs/

analysis_samples.txt   pheno_cov.tsv		     vcf_samples.txt
covariate_columns.txt  selected_phenotype_stats.tsv
flare_uris.json        selected_phenotypes.txt


In [4]:
select_py = SCRIPTS / "select_phenotypes.py"
assert select_py.exists(), select_py

sh(
    f"python3 {select_py} "
    f"--phenotype-csv {PHENOTYPE_CSV} "
    f"--covariates-csv {COVARIATES_CSV} "
    f"--vcf-samples {vcf_samples} "
    f"--out-dir {OUT_DIR} "
    f"--n-phenotypes {N_PHENOTYPES} "
    f"--min-cases {MIN_CASES} "
    f"--n-pcs {N_PCS}"
)

stats = pd.read_csv(OUT_DIR / "selected_phenotype_stats.tsv", sep="\t")
display(stats)
pheno_cov = pd.read_csv(OUT_DIR / "pheno_cov.tsv", sep="\t", nrows=5)
display(pheno_cov)
limited_matrix = pd.read_csv(OUT_DIR / "covariates_limited.tsv", sep="\t", nrows=5)
full_matrix = pd.read_csv(OUT_DIR / "covariates_full.tsv", sep="\t", nrows=5)
display(limited_matrix)
display(full_matrix)
print("limited covariates:", (OUT_DIR / "covariate_columns_limited.txt").read_text().split())
print("full covariates:", (OUT_DIR / "covariate_columns_full.txt").read_text().split())
print("legacy covariate_columns:", (OUT_DIR / "covariate_columns.txt").read_text().split())
levels = pd.read_csv(OUT_DIR / "technical_covariate_levels.tsv", sep="\t")
display(levels)
n_samples = sum(1 for _ in open(OUT_DIR / "analysis_samples.txt"))
print(f"shared full-complete analysis samples: {n_samples:,}")
print(f"num_ancs (for WDL inputs): {NUM_ANCS}")

python3 /home/jupyter/AoU_DRC_LongReads_PhaseTwo_Storage/scripts/select_phenotypes.py --phenotype-csv AoU_Phase2_Phenotype.csv.gz --covariates-csv covariates.v2.csv.gz --vcf-samples tractor_mix_pilot_inputs/vcf_samples.txt --out-dir tractor_mix_pilot_inputs --n-phenotypes 10 --min-cases 100 --n-pcs 10
VCF samples: 12,347
Dropped withdrawn covariates rows: 134
Intersected analysis samples (VCF order): 9,851 (vcf=12,347, pheno=9,954, cov=12,554)
Dropped 9851 samples missing full-model covariates; remaining 0
No samples remain after requiring full-model covariate completeness


,phenotype,safe_name,n_cases,n_controls,n_missing
0,CV_401,CV_401,3041,6238,572
1,CV_401.1,CV_401_1,2928,6360,563
2,MS_713,MS_713,2824,6103,924
3,EM_239,EM_239,2777,6505,569
4,SS_809,SS_809,2773,5926,1152
5,MS_713.3,MS_713_3,2729,6193,929
6,ID_089,ID_089,2688,5893,1270
7,MS_718,MS_718,2634,6246,971
8,GI_527,GI_527,2351,6503,997
9,EM_236,EM_236,2056,7109,686


,ID,sex,age,PC1,PC2,PC3,PC4,PC5,PC6,PC7,...,CV_401,CV_401_1,MS_713,EM_239,SS_809,MS_713_3,ID_089,MS_718,GI_527,EM_236


FileNotFoundError: [Errno 2] No such file or directory: 'tractor_mix_pilot_inputs/covariates.limited.txt'

In [ ]:
# Upload prep outputs to the workspace bucket for Terra / Cromwell
samples_path = OUT_DIR / "analysis_samples.txt"
n_upload = sum(1 for line in samples_path.read_text().splitlines() if line.strip())
assert n_upload > 1000, (
    f"Refusing upload: {samples_path} has only {n_upload} IDs "
    f"({samples_path.stat().st_size} bytes). Re-run select_phenotypes first."
)

if ws:
    dest = f"{ws}/tractor_mix_pilot/"
    for name in [
        "analysis_samples.txt",
        "pheno_cov.tsv",
        "covariates_limited.tsv",
        "covariates_full.tsv",
        "selected_phenotypes.txt",
        "selected_phenotype_stats.tsv",
        "covariate_columns_limited.txt",
        "covariate_columns_full.txt",
        "covariate_columns.txt",
        "technical_covariate_levels.tsv",
        "vcf_samples.txt",
        "flare_uris.json",
    ]:
        path = OUT_DIR / name
        if path.exists():
            sh(f"gsutil cp {path} {dest}{name}")
    sh(f"gsutil ls -l {dest}analysis_samples.txt")
    sh(f"gsutil cat {dest}analysis_samples.txt | wc -l")
    print("Uploaded to", dest)
    print("Use the same shared cohort for all four matched runs:")
    print(f"  analysis_samples:        {dest}analysis_samples.txt")
    print(f"  pheno_cov:               {dest}pheno_cov.tsv")
    print(f"  selected_phenotypes:     {dest}selected_phenotypes.txt")
    print(f"  covariates_limited:      {dest}covariates_limited.tsv")
    print(f"  covariates_full:         {dest}covariates_full.tsv")
    print(f"  columns (limited):       {dest}covariate_columns_limited.txt")
    print(f"  columns (full):          {dest}covariate_columns_full.txt")
    print(f"  technical levels:        {dest}technical_covariate_levels.tsv")
    print("Tractor-Mix / SAIGE: use pheno_cov.tsv plus the matching covariate_columns file.")
else:
    print("WORKSPACE_BUCKET unset; outputs left in", OUT_DIR.resolve())